In [4]:
from pathlib import Path

from ensemble import compute_rmwd

%load_ext autoreload
%autoreload 2

systems = ['6h86_A', '6p5h_A', '6jv8_A', '7lp1_A', '7jfl_C']

md_folder = Path('/home/lcl35/project_pi_sk2433/lcl35/ProtSCAPE-Net/data/datasets/')
ensemble_folder = Path('/home/lcl35/project_pi_sk2433/lcl35/ProtSCAPE-Net/Generation/ATLAS_1K_traj')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
for system in systems:

    md_topo = md_folder / f'{system}_protein' / f'{system}.pdb'
    md_traj = md_folder / f'{system}_protein' / f'{system}_prod_R1_fit.xtc'

    gen_topo = ensemble_folder / system / f'{system}.pdb'
    gen_traj = ensemble_folder / system / f'{system}.xtc'

    !python compute_metrics.py \
        --system {system} \
        --ref_topo {md_topo} \
        --ref_traj {md_traj} \
        --pred_topo {gen_topo} \
        --pred_traj {gen_traj} 

/home/lcl35/.conda/envs/protscape/lib/python3.12/site-packages/MDAnalysis/analysis/dihedrals.py:444: UserWarning: Cannot determine phi and psi angles for the first or last residues
  warnings.warn(
/home/lcl35/.conda/envs/protscape/lib/python3.12/site-packages/MDAnalysis/analysis/dihedrals.py:574: DeprecationWarning: The `angle` attribute was deprecated in MDAnalysis 2.0.0 and will be removed in MDAnalysis 3.0.0. Please use `results.angles` instead
  warnings.warn(wmsg, DeprecationWarning)
/home/lcl35/.conda/envs/protscape/lib/python3.12/site-packages/MDAnalysis/analysis/rms.py:1000: DeprecationWarning: The `rmsf` attribute was deprecated in MDAnalysis 2.0.0 and will be removed in MDAnalysis 3.0.0. Please use `results.rmsd` instead.
  warnings.warn(wmsg, DeprecationWarning)
/home/lcl35/.conda/envs/protscape/lib/python3.12/site-packages/MDAnalysis/analysis/dihedrals.py:444: UserWarning: Cannot determine phi and psi angles for the first or last residues
  warnings.warn(
/home/lcl35/.cond

In [2]:
for system in systems:

    pdb, chain = system.split('_')

    md_topo = md_folder / f'{system}_protein' / f'{system}.pdb'
    md_traj = md_folder / f'{system}_protein' / f'{system}_prod_R1_fit.xtc'

    reps = [md_folder / f'{system}_protein' / f'{system}_prod_R{i}_fit.xtc' for i in (2, 3)]

    gen_topo = ensemble_folder / pdb / f'{pdb}_topo.pdb'
    gen_traj = ensemble_folder / pdb / f'{pdb}_traj.xtc'

    ap_traj = alphaflow_folder / pdb / f'{pdb}_traj.xtc'
    ap_topo = alphaflow_folder / pdb / f'{pdb}_topo.pdb'

    rmwd = compute_rmwd(md_topo, md_traj, gen_topo, gen_traj, 
                    align_to=md_topo, align_sel="protein and name CA", point_sel="protein and name CA")
    
    print(f"{system}: RMWD = {rmwd['rmwd']:.3f} Å (translation: {rmwd['translation']:.3f} Å, variance: {rmwd['variance']:.3f} Å)")

    ap_rmwd = compute_rmwd(md_topo, md_traj, ap_topo, ap_traj, 
                    align_to=md_topo, align_sel="protein and name CA", point_sel="protein and name CA")
    
    print(f"  AlphaFlow: RMWD = {ap_rmwd['rmwd']:.3f} Å (translation: {ap_rmwd['translation']:.3f} Å, variance: {ap_rmwd['variance']:.3f} Å)")

    # for i, rep in enumerate(reps):
    #     rmwd_rep = compute_rmwd(md_topo, rep, gen_topo, gen_traj, 
    #                         align_to=md_topo, align_sel="protein and name CA", point_sel="protein and name CA")
    #     print(f"  Rep {i+2}: RMWD = {rmwd_rep['rmwd']:.3f} Å (translation: {rmwd_rep['translation']:.3f} Å, variance: {rmwd_rep['variance']:.3f} Å)")



6h86_A: RMWD = 0.697 Å (translation: 0.426 Å, variance: 0.551 Å)
  AlphaFlow: RMWD = 12.924 Å (translation: 12.424 Å, variance: 3.561 Å)
6p5h_A: RMWD = 0.342 Å (translation: 0.231 Å, variance: 0.251 Å)
  AlphaFlow: RMWD = 10.982 Å (translation: 9.857 Å, variance: 4.843 Å)
6jv8_A: RMWD = 0.603 Å (translation: 0.458 Å, variance: 0.392 Å)
  AlphaFlow: RMWD = 2.059 Å (translation: 1.873 Å, variance: 0.854 Å)
7lp1_A: RMWD = 0.539 Å (translation: 0.396 Å, variance: 0.367 Å)
  AlphaFlow: RMWD = 2.031 Å (translation: 1.838 Å, variance: 0.864 Å)
7jfl_C: RMWD = 0.430 Å (translation: 0.364 Å, variance: 0.230 Å)
  AlphaFlow: RMWD = 3.578 Å (translation: 3.037 Å, variance: 1.892 Å)


In [3]:
from ensemble import subsampled_rmwd

for system in systems:

    pdb, chain = system.split('_')

    md_topo = md_folder / f'{system}_protein' / f'{system}.pdb'
    md_traj = md_folder / f'{system}_protein' / f'{system}_prod_R1_fit.xtc'

    results = subsampled_rmwd(md_topo, md_traj, point_sel="protein and name CA", trials=20, subsampling=100)
    print(f"{system}: RMWD = {results['rmwd_mean']:.3f} +- {results['rmwd_std']:.3f}" )


6h86_A: RMWD = 0.627 +- 0.177
6p5h_A: RMWD = 0.384 +- 0.095
6jv8_A: RMWD = 0.321 +- 0.083
7lp1_A: RMWD = 0.272 +- 0.054
7jfl_C: RMWD = 0.289 +- 0.089


In [3]:
from ensemble import  plot_pca_space, compute_pca_dist

for system in systems:
    
    pdb, chain = system.split('_')

    md_topo = md_folder / f'{system}_protein' / f'{system}.pdb'
    md_traj = md_folder / f'{system}_protein' / f'{system}_prod_R1_fit.xtc'

    gen_topo = ensemble_folder / pdb / f'{pdb}_topo.pdb'
    gen_traj = ensemble_folder / pdb / f'{pdb}_traj.xtc'

    ap_topo = alphaflow_folder / pdb / f'{pdb}_topo.pdb'
    ap_traj = alphaflow_folder / pdb / f'{pdb}_traj.xtc'

    reps = [md_folder / f'{system}_protein' / f'{system}_prod_R{i}_fit.xtc' for i in (2, 3)]


    # ca_w2 = compute_pca_dist(
    #     top1=md_topo,
    #     traj1=md_traj,
    #     top2=gen_topo,
    #     traj2=gen_traj,
    #     mode="ca",
    #     point_sel="protein and name CA",
    #     pca_mode="ref",
    #     k=2,
    #     seed=42,
    #     align_to=md_topo,  # align both to MD topology/frame
    #     align_sel="protein and name CA",
    #     metric="empirical_w2",
    # )

    # print(f"{system}: PCA W2 distance = {ca_w2['w2']:.3f}")
    # plot_pca_space(ca_w2, alpha=0.5, s=20, show_gaussian=False)

    af_w2 = compute_pca_dist(
        top1=md_topo,
        traj1=md_traj,
        top2=ap_topo,
        traj2=ap_traj,
        mode="ca",
        point_sel="protein and name CA",
        pca_mode="ref",
        k=2,
        seed=42,
        align_to=md_topo,  # align both to MD topology/frame
        align_sel="protein and name CA",
        metric="empirical_w2",
    )

    print(f"  AlphaFlow: PCA W2 distance = {af_w2['w2']:.3f}")
    # plot_pca_space(af_w2, alpha=0.5, s=20, show_gaussian=False)

    # for i, rep in enumerate(reps):
    #     ca_w2_rep = compute_pca_dist(
    #         top1=md_topo,
    #         traj1=md_traj,
    #         top2=md_topo,  # use MD topology for rep as well
    #         traj2=rep,
    #         mode="ca",
    #         point_sel="protein and namd CA",
    #         pca_mode="ref",
    #         k=2,
    #         seed=42,
    #         align_to=md_topo,  # align both to MD topology/frame
    #         align_sel="protein and name CA",
    #         metric="empirical_w2",
    #     )
    #     print(f"  Rep {i+2}: PCA W2 distance = {ca_w2_rep['w2']:.3f}")
        # plot_pca_space(ca_w2_rep, alpha=0.5, s=20, show_gaussian=False)


    

(10001, 264) (100, 264)


/home/lcl35/.conda/envs/protscape/lib/python3.12/site-packages/ot/lp/_network_simplex.py:574: UserWarning: numItermax reached before optimality. Try to increase numItermax.
  check_result(result_code)


  AlphaFlow: PCA W2 distance = 96.138
(10001, 315) (100, 315)
  AlphaFlow: PCA W2 distance = 35.507
(10001, 228) (100, 228)
  AlphaFlow: PCA W2 distance = 10.534
(10001, 117) (100, 117)
  AlphaFlow: PCA W2 distance = 9.513
(10001, 141) (100, 141)
  AlphaFlow: PCA W2 distance = 14.553


In [5]:
from ensemble import compute_pc_similarity

for system in systems:
    
    pdb, chain = system.split('_')

    md_topo = md_folder / f'{system}_protein' / f'{system}.pdb'
    md_traj = md_folder / f'{system}_protein' / f'{system}_prod_R1_fit.xtc'

    gen_topo = ensemble_folder / pdb / f'{pdb}_topo.pdb'
    gen_traj = ensemble_folder / pdb / f'{pdb}_traj.xtc'

    ap_topo = alphaflow_folder / pdb / f'{pdb}_topo.pdb'
    ap_traj = alphaflow_folder / pdb / f'{pdb}_traj.xtc'



    pc = compute_pc_similarity(
        top1=md_topo,
        traj1=md_traj,
        top2=gen_topo,
        traj2=gen_traj,
        mode="ca",
        point_sel="protein and name CA",
        align_to=md_topo,  # align both to MD topology/frame
        align_sel="protein and name CA",
        threshold=0.5,
        seed=0)

    print(f"{system}: PC similarity = {pc['pc_sim_percent']:.2f}% of PCs > 0.5 using {pc['m']} PCs")

    ap_pc = compute_pc_similarity(
        top1=md_topo,
        traj1=md_traj,
        top2=ap_topo,
        traj2=ap_traj,
        mode="ca",
        point_sel="protein and name CA",
        align_to=md_topo,  # align both to MD topology/frame
        align_sel="protein and name CA",
        threshold=0.5,
        seed=0)

    print(f"  AlphaFlow: PC similarity = {ap_pc['pc_sim_percent']:.2f}% of PCs > 0.5 using {ap_pc['m']} PCs")
        

6h86_A: PC similarity = 12.12% of PCs > 0.5 using 99 PCs
  AlphaFlow: PC similarity = 1.01% of PCs > 0.5 using 99 PCs
6p5h_A: PC similarity = 15.15% of PCs > 0.5 using 99 PCs
  AlphaFlow: PC similarity = 1.01% of PCs > 0.5 using 99 PCs
6jv8_A: PC similarity = 11.11% of PCs > 0.5 using 99 PCs
  AlphaFlow: PC similarity = 1.01% of PCs > 0.5 using 99 PCs
7lp1_A: PC similarity = 9.09% of PCs > 0.5 using 99 PCs
  AlphaFlow: PC similarity = 0.00% of PCs > 0.5 using 99 PCs
7jfl_C: PC similarity = 9.09% of PCs > 0.5 using 99 PCs
  AlphaFlow: PC similarity = 0.00% of PCs > 0.5 using 99 PCs


In [1]:
for system in systems:
    
    pdb, chain = system.split('_')

    md_topo = md_folder / f'{system}_protein' / f'{system}.pdb'
    md_traj = md_folder / f'{system}_protein' / f'{system}_prod_R1_fit.xtc'

    gen_topo = ensemble_folder / f'{pdb}_topo.pdb'
    gen_traj = ensemble_folder / f'{pdb}_traj.xtc'

    reps = [md_folder / f'{system}_protein' / f'{system}_prod_R{i}_fit.xtc' for i in (2, 3)]


    jsd = compute_pca_dist(
        top1=md_topo,
        traj1=md_traj,
        top2=gen_topo,
        traj2=gen_traj,
        mode="ca",
        point_sel="protein and backbone",
        pca_mode="pool",
        k=2,
        seed=42,
        align_to=md_topo,  # align both to MD topology/frame
        align_sel="protein and name CA",
        metric="jsd",
    )

    print(f"{system}: PCA JSD = {jsd['jsd']:.3f}")

    for i, rep in enumerate(reps):
        jsd_rep = compute_pca_dist(
            top1=md_topo,
            traj1=rep,
            top2=gen_topo,
            traj2=gen_traj,
            mode="ca",
            point_sel="protein and backbone",
            pca_mode="pool",
            k=2,
            seed=42,
            align_to=md_topo,  # align both to MD topology/frame
            align_sel="protein and name CA",
            metric="jsd",
        )
        print(f"  Rep {i+2}: PCA JSD = {jsd_rep['jsd']:.3f}")

    plot_pca_space(jsd, alpha=0.5, s=20, show_gaussian=True)

NameError: name 'systems' is not defined